In [ ]:
# 必要パッケージ
!pip install datasets==3.6.0
!pip install farm-haystack[elasticsearch]

In [ ]:
## データセット用意
from datasets import load_dataset
subjqa = load_dataset("subjqa", name="electronics")
import pandas as pd
dfs = {split: dset.to_pandas() for split, dset in subjqa.flatten().items()}

In [ ]:
## Elasticsearch ダウンロード
url = "https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-7.17.10-linux-x86_64.tar.gz"
!wget -nc -q {url}
!tar -xzf elasticsearch-7.17.10-linux-x86_64.tar.gz

In [ ]:
## Elasticsearch 起動
!chown -R daemon:daemon /content/elasticsearch-7.17.10
!su -s /bin/bash -c 'export ES_JAVA_OPTS="-Des.cgroups.hierarchy.override=/ -XX:-UseContainerSupport"; \
/content/elasticsearch-7.17.10/bin/elasticsearch \
  -Ediscovery.type=single-node \
  -Expack.security.enabled=false \
  -Expack.ml.enabled=false \
  -Ehttp.host=127.0.0.1 \
  -Etransport.host=127.0.0.1 \
  > /tmp/es.log 2>&1 &' daemon

In [ ]:
## (少し待ってから)起動確認
!curl -X GET "localhost:9200/?pretty"

In [ ]:
## Retriever の用意
### ドキュメントストアのインスタンス化
from haystack.document_stores import ElasticsearchDocumentStore
document_store = ElasticsearchDocumentStore(return_embedding=True)

### ドキュメントストアのインデックス化
for split, df in dfs.items():
  # 重複するレビュー除去
  docs = [{"content": row["context"],
           "meta": {
               "item_id": row["title"],
               "question_id": row["id"],
               "split": split
           }}
          for _, row in df.drop_duplicates(subset="context").iterrows()
         ]
  document_store.write_documents(docs, index="document")

### Elasticsearch を用いたレトリーバー
from haystack.nodes.retriever.sparse import BM25Retriever
es_retriever = BM25Retriever(document_store=document_store)

In [ ]:
# パイプライン構築
from haystack.pipelines import Pipeline
# from haystack.eval